# Nedbrytning av variation i skadereglering längs ett försäkringsbolags organisationshierarki med PROC NESTED

## Sammanfattning

Ett sak- och motorfordonsförsäkringsbolag vill veta **var** inkonsekvensen i skaderegleringen uppstår: drivs den av skillnader mellan geografiska regioner, mellan kontor, mellan enskilda handläggare, eller helt enkelt av slumpmässigt brus mellan skador? Denna notebook bygger upp en syntetisk, fullständigt nästlad bilskadeportfölj (Region > Kontor > Handläggare > skada) och använder **PROC NESTED** för att utföra en slumpeffekts-variansanalys, där variansbidraget från varje nivå i hierarkin skattas och redovisas som procent av totalen. Resultatet talar om för ledningen vilken organisatorisk nivå som bör prioriteras för att göra skaderegleringen mer konsekvent.

## Datakällor

All data genereras inline i det första DATA-steget — inga externa filer, inget nätverk. Designen är **fullständigt nästlad**: varje kontor tillhör exakt en region, varje handläggare tillhör exakt ett kontor, och varje skada tillhör exakt en handläggare.

| Datamängd | Rader | Variabel | Typ | Roll | Beskrivning |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (nivå 1, yttersta) | Geografisk region (5 regioner: NO, SO, ML, VK, SV) |
| | | `Branch` | char | CLASS (nivå 2, nästlad i Region) | Kontorskod (2 kontor per region) |
| | | `Adjuster` | char | CLASS (nivå 3, nästlad i Branch) | Handläggar-ID (2 handläggare per kontor) |
| | | `Settlement` | num | VAR (beroende) | Utbetald ersättning för bilskadan, i USD |
| | | `CycleDays` | num | VAR (beroende) | Antal dagar från skadeanmälan till reglering |

Syntetisk struktur: 5 regioner x 2 kontor x 2 handläggare x 2 skador = 40 observationer. En regional slumpeffekt, en kontorseffekt nästlad i region, en handläggareffekt nästlad i kontor, och en residual på skadenivå läggs samman additivt via `rand('NORMAL', ...)` så att variankomponenterna kan återvinnas. Regionseffekten dras med störst standardavvikelse (2 200) så att *var* en skada hanteras blir den dominerande drivkraften. Fröet är fixerat med `call streaminit(20260531)`. En kompakt, fullt balanserad design håller varje nivå i hierarkin befolkad med reella frihetsgrader.

# Nedbrytning av variation i skadereglering med PROC NESTED

När ett sak- och motorfordonsförsäkringsbolag granskar hur mycket det betalar för att reglera bilskador upptäcker det ofta stor variation. En del av variationen är oundviklig (varje olycka är unik), men en del av den återspeglar **organisatoriska** faktorer — en region betalar rikligare än en annan, ett kontor är mer generöst, en enskild handläggare avviker.

Data har en strikt **nästlad** (hierarkisk) struktur:

```
Region  ->  Kontor (nästlat i Region)  ->  Handläggare (nästlad i Kontor)  ->  enskilda skador
```

Ett kontor tillhör exakt en region och en handläggare tillhör exakt ett kontor, så faktorerna är nästlade snarare än korsade. `PROC NESTED` utför en slumpeffekts-variansanalys för exakt denna design och skattar en **variankomponent** på varje nivå, uttryckt som procent av totalen. Den procentuella fördelningen besvarar affärsfrågan: *vilken organisationsnivå bör vi prioritera för att göra skaderegleringen mer konsekvent?*

## Steg 1 — Generera en syntetisk, fullständigt nästlad skadeportfölj

Vi simulerar 5 regioner, var och en med 2 kontor, var och en med 2 handläggare, var och en hanterande 2 skador (40 rader totalt). Vi bygger upp utfallet från additiva slumpeffekter så att varje nivå verkligen bidrar med varians:

- en **regional** effekt (störst spridning — regionerna skiljer sig mest),
- en **kontorseffekt nästlad i region**,
- en **handläggareffekt nästlad i kontor**,
- och en **residual på skadenivå** (bruset inom handläggaren).

`call streaminit` låser fröet för reproducerbarhet; varje effekt dras med `rand('NORMAL', medelvärde, sd)`. Regionseffekten använder den bredaste standardavvikelsen så att den bör ta den största andelen av totalvariansen. Vi genererar även ett andra utfall, `CycleDays`, som delar samma hierarki så att vi senare kan visa analysen med flera utfall.

In [1]:
data claims;
   CALL streaminit(20260531);
   LÄNGD Region $2 Branch $6 Adjuster $9;

   /* Regional slumpeffekt: regionerna skiljer sig mest. */
   GÖR r = 1 TILL 5;
      Region = SCAN('NO SO ML VK SV', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Kontor nästlat i region. */
      GÖR b = 1 TILL 2;
         Branch = catx('-', Region, put(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Handläggare nästlad i kontor. */
         GÖR a = 1 TILL 2;
            Adjuster = catx('-', Branch, put(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Enskilda skador hanterade av denna handläggare. */
            GÖR claim = 1 TILL 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               OM Settlement < 0 SÅ Settlement = 0;
               OM CycleDays  < 1 SÅ CycleDays  = 1;
               UTDATA;
            SLUT;
         SLUT;
      SLUT;
   SLUT;

   BEHÅLL Region Branch Adjuster Settlement CycleDays;
KÖR;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Steg 2 — Sortera efter nästlingshierarkin

`PROC NESTED` kräver att indata är **sorterade efter CLASS-variablerna i den ordning de anges** — yttersta faktorn först. Vi sorterar efter `Region Branch Adjuster` så att proceduren kan gå igenom hierarkin korrekt.

In [2]:
PROC SORT data=claims;
   EFTER Region Branch Adjuster;
KÖR;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Steg 3 — Variankomponentanalys av ersättningsbeloppet

Den centrala analysen. Vi listar CLASS-variablerna från yttersta till innersta (`Region Branch Adjuster`); den innersta replikeringen — de enskilda skadorna — behandlas automatiskt som feltermen. `VAR Settlement` anger det enda utfallet.

`LABEL`-satsen ger ett mer läsvänligt visningsnamn för utfallet i utdatans rubrik. Utdata ger koefficienterna för förväntade medelkvadratsummor, variansanalystabellen (med ett F-test av varje variankomponent mot noll), samt skattningen av **variankomponenten** på varje nivå tillsammans med dess **procent av totalen**.

In [3]:
TITEL "Variansdekomponering av bilskadeersättningar";
title2 "Region/kontor/handläggare slumpeffekts-ANOVA";

PROC NESTED data=claims;
   KLASS Region Branch Adjuster;
   VARIABEL Settlement;
   ETIKETT Region="Region"
           Branch="Kontor"
           Adjuster="Handläggare"
           Settlement="Utbetald ersättning (USD)";
KÖR;


                                      Variansdekomponering av bilskadeersättningar                                      
                                      Region/kontor/handläggare slumpeffekts-ANOVA                                      


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Utbetald ersättning (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Kontor  15528530.531717  1651824.844989             54.4057      8.0000
Kontor                 5  11569658.859028          1.89      0.1829  Handläggare  2313931.771806   272682.828942              8.9813      4.0000
Handläggare           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200006


NOTE: Option TITLE changed to Variansdekomponering av bilskadeersättningar.
NOTE: Option TITLE2 changed to Region/kontor/handläggare slumpeffekts-ANOVA.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Steg 4 — Analysera två utfall samtidigt

Ledningen bryr sig även om **handläggningstiden** — hur lång tid det tar att reglera skador. Vi lägger till `CycleDays` i `VAR`-listan. Med mer än en beroende variabel redovisar `PROC NESTED` dessutom en **kovariationsanalys** som visar hur de två utfallen samvarierar på varje nivå i hierarkin (till exempel om regionerna som betalar mer också reglerar långsammare).

In [4]:
TITEL "Ersättnings- och handläggningstidens variankomponenter";
title2 "Två utfallsvariabler i samma nästlade hierarki";

PROC NESTED data=claims;
   KLASS Region Branch Adjuster;
   VARIABEL Settlement CycleDays;
   ETIKETT Region="Region"
           Branch="Kontor"
           Adjuster="Handläggare"
           Settlement="Utbetald ersättning (USD)"
           CycleDays="Dagar till reglering";
KÖR;


                                 Ersättnings- och handläggningstidens variankomponenter                                 
                                     Två utfallsvariabler i samma nästlade hierarki                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Utbetald ersättning (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Kontor  15528530.531717  1651824.844989             54.4057      8.0000
Kontor                 5  11569658.859028          1.89      0.1829  Handläggare  2313931.771806   272682.828942              8.9813      4.0000
Handläggare           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200006


NOTE: Option TITLE changed to Ersättnings- och handläggningstidens variankomponenter.
NOTE: Option TITLE2 changed to Två utfallsvariabler i samma nästlade hierarki.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Steg 5 — Enbart variansvy med alternativet AOV

För en snabb sammanfattning av variankomponenterna för båda utfallen begränsar alternativet `AOV` utdata till variansanalysstatistiken och **undertrycker avsnittet för kovariationsanalys**. Detta är den kompakta vy en portföljanalytiker skulle skumma när de bara behöver variansfördelningen per nivå för varje utfall, inte kovariationen mellan utfallen.

In [5]:
TITEL "Endast variankomponenter (AOV)";

PROC NESTED data=claims aov;
   KLASS Region Branch Adjuster;
   VARIABEL Settlement CycleDays;
   ETIKETT Region="Region"
           Branch="Kontor"
           Adjuster="Handläggare"
           Settlement="Utbetald ersättning (USD)"
           CycleDays="Dagar till reglering";
KÖR;

TITEL;


                                             Endast variankomponenter (AOV)                                             
                                     Två utfallsvariabler i samma nästlade hierarki                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Utbetald ersättning (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Kontor  15528530.531717  1651824.844989             54.4057      8.0000
Kontor                 5  11569658.859028          1.89      0.1829  Handläggare  2313931.771806   272682.828942              8.9813      4.0000
Handläggare           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200006


NOTE: Option TITLE changed to Endast variankomponenter (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Tolkning av resultaten

Kolumnen **Percent of Total** i variansanalystabellen är huvudresultatet. Läser man den uppifrån och ned får man andelen av den totala ersättningsvariationen som kan tillskrivas varje organisatorisk nivå. För ersättningsbeloppet ger körningen:

- **Region — 54,4 %** (Pr > F = 0,0304). Data genererades med den största regionala spridningen, och analysen återfinner den: mer än hälften av all variation i ersättningsbelopp finns *mellan* regioner — statistiskt signifikant belägg för att *var* en skada hanteras är den dominerande drivkraften.
- **Kontor (inom Region) — 9,0 %** (Pr > F = 0,18). En måttlig ytterligare andel när man går från region ner till enskilt kontor; inte signifikant här.
- **Handläggare (inom Kontor) — 3,7 %** (Pr > F = 0,33). Lite variation kan spåras till den enskilda handläggaren i denna portfölj.
- **Error — 32,9 %.** Den kvarvarande skada-till-skada-bruset inom en handläggare; detta är den irreducibla variation som ingen organisatorisk åtgärd kan ta bort.

Varje nivå bär också ett **F-test (Pr > F)** av nollhypotesen att dess variankomponent är noll. Det låga p-värdet för Region (0,0304) är statistiskt belägg för genuina systematiska skillnader mellan regioner, inte slumpmässig variation.

Handläggningstidens utfall berättar en parallell historia: **Region svarar för 45,8 %** av variationen i dagar-till-reglering (Pr > F = 0,0448, återigen signifikant), med Kontor och Handläggare bidragande med enstaka procentandelar och residualen på 40,1 %. Så både *hur mycket* ett försäkringsbolag betalar och *hur lång tid* det tar styrs först och främst av region.

Körningen med två utfall lägger till en **kovariationsanalys**. Här driver Region-nivån korsprodukterna, och den övergripande **korrelationskoefficienten är -0,36** — ett negativt samband: regionerna som betalar större ersättningar tenderar att *reglera dem snabbare*, inte långsammare. Det är ett användbart, icke uppenbart fynd — de dyra regionerna är inte de långsamma.

**Affärsslutsats:** eftersom Region dominerar procentfördelningen för båda utfallen bör försäkringsbolaget standardisera regleringsriktlinjer och befogenhetsgränser *mellan regioner* först — det är där den största, statistiskt signifikanta inkonsekvensen finns — innan man investerar i insatser på kontors- eller handläggarnivå. Det negativa sambandet mellan ersättning och handläggningstid innebär att ingen enskild region är både dyrast och långsammast, så de två problemen kan angripas med separata, regionriktade program. PROC NESTED omvandlar en vag känsla av "vår skadereglering är inkonsekvent" till en konkret, nivå-för-nivå-tillskrivning av var den inkonsekvensen finns.